# UltraDES → ESP32 Supervisor Generator

This notebook builds the **`supervisor_data_<PROBLEM>.h`** file consumed by the companion Arduino sketch `homomorphic-esp32-ultrades.ino`. The sketch evaluates Discrete Event System supervisors **homomorphically** on an ESP32 — the plant state stays encrypted end-to-end and is never decrypted on the run path.

## How to use

1. **Cell 3** — set `PROBLEM` to the problem you want to evaluate.
2. **Run all cells** top-to-bottom (Kernel → Restart & Run All).
3. **Copy the generated `supervisor_data_<PROBLEM>.h`** into your Arduino sketch folder, alongside `sdkconfig.ext`.
4. In the `.ino`, update the `#include` line to match the new filename.
5. Flash and open Serial Monitor at 115200 baud.

## Problems included

| `PROBLEM` value | Description |
|---|---|
| `small_factory` | M1 → Buffer → M2 (simplest case, 1 supervisor) |
| `extended_small_factory` | M1 → B1 → M2 → B2 → M3 (2 supervisors) |
| `fms` | Flexible Manufacturing System — Queiroz & Cury 2000 (7 supervisors, S6 has 164 states) |

## Adding your own problem

In **Cell 4**, define a `build_<your_name>()` function following the pattern of the existing three. It must return `(plants, specs, spec_groups, sim_seq)`:

- **`plants`** — list of plant DFAs (the physical model)
- **`specs`** — list of specification DFAs (the desired behaviour)
- **`spec_groups`** — list of lists; groups specs that share events through the plant. Specs in *different* groups must have disjoint alphabets.
- **`sim_seq`** — list of event-name strings — the sequence the ESP32 benchmark will run.

Then add the function to the `PROBLEMS` dict at the bottom of Cell 4 and set `PROBLEM = '<your_name>'` in Cell 3.

## What gets generated

A single file is written next to this notebook: **`supervisor_data_<PROBLEM>.h`** — direct C arrays in PROGMEM. No JSON, no runtime parsing on the ESP32. Contains the monolithic supervisor (when it fits in flash) plus the local-modular and reduced supervisors.

The output is **reproducible**: re-running the notebook produces byte-identical output (Cell 5 sorts UltraDES's otherwise non-deterministic transition iteration order).

## Requirements

- Python 3.8+
- .NET Runtime 8.0
- [UltraDES-Python](https://github.com/lacsed/UltraDES-Python)

Cell 1 installs everything automatically on **Linux** / **Google Colab**. On Windows or macOS, Cell 1 detects the OS and prints the manual installation command — you'll need to install .NET 8 yourself first.


In [ ]:
# ── Cell 1: Install UltraDES ─────────────────────────────────────────────────
# Installs the .NET runtime and the UltraDES-Python wrapper.
# Safe to re-run — pip skips already-installed packages.
#
# NOTE: this cell uses `apt-get`, so it only works on Linux / Google Colab.
# On Windows or macOS you must install .NET 8 manually first
# (https://dotnet.microsoft.com/download) and then run only the pip line below.

import platform
from IPython.display import clear_output

if platform.system() != 'Linux':
    print(f"WARNING: this cell uses apt-get and only works on Linux / Colab.")
    print(f"Detected: {platform.system()}.  Install .NET 8 manually, then run:")
    print(f"  pip install https://github.com/lacsed/UltraDES-Python/releases/download/0.0.6/ultrades_python-0.0.6-py3-none-any.whl")
else:
    !sudo apt-get install -y dotnet-runtime-8.0
    !pip install --no-cache-dir https://github.com/lacsed/UltraDES-Python/releases/download/0.0.6/ultrades_python-0.0.6-py3-none-any.whl
    clear_output()
    print("UltraDES installed.")


In [ ]:
# ── Cell 2: Imports ───────────────────────────────────────────────────────────
# Load the .NET / pythonnet bridge and import UltraDES automata primitives.

import sys, os, json

from pythonnet import load
load("coreclr")

import ultrades
sys.path.append(os.path.dirname(ultrades.__file__))

from ultrades.automata import (
    state,                    # create a DFA state
    event,                    # create an event (controllable or not)
    dfa,                      # create a DFA from a transition list
    monolithic_supervisor,    # compute the monolithic supremal supervisor
    local_modular_supervisors,          # compute local modular supervisors
    local_modular_reduced_supervisors,  # compute reduced local modular supervisors
    parallel_composition,     # synchronous parallel composition of two DFAs
)

print("UltraDES imported successfully.")


<IPython.core.display.Javascript object>

UltraDES imported successfully.


In [ ]:
# ── Cell 3: Choose problem ────────────────────────────────────────────────────
# This is the ONLY cell you need to edit between runs.
# Set PROBLEM to one of the values listed in the README above.

PROBLEM = 'delivery_system'   # <── change this line

# Valid options:
#   'small_factory'           simplest two-machine pipeline
#   'extended_small_factory'  three-machine pipeline
#   'fms'                     Flexible Manufacturing System
#   'delivery_system'         Cargo-transfer UAV system (D1->B->D2/D3)

print(f"Problem selected: {PROBLEM}")


Problem selected: small_factory


In [ ]:
# ── Cell 4: Problem definitions ───────────────────────────────────────────────
#
# Each builder function returns a tuple:
#   (plants, specs, spec_groups, sim_seq)
#
#   plants      — list of plant DFAs (model the physical machines/buffers)
#   specs       — list of specification DFAs (model the desired behaviour)
#   spec_groups — list of lists grouping specs that share events.
#                 Each group is synthesised independently for local modular.
#                 Specs in the SAME group share events through the plant and
#                 must be grouped to avoid "Conflicting Supervisors" errors.
#                 Specs in DIFFERENT groups have disjoint alphabets.
#   sim_seq     — list of event names for the simulation sequence used in
#                 the ESP32 benchmark
#
# IMPORTANT — UltraDES event matching:
#   UltraDES synchronises automata by event NAME, not by object identity.
#   Plants and specs must use separate event objects with the same name string
#   so that UltraDES can synchronise them during composition.
#   Example:  e2 = event('e2', False)  in the plant
#             e2_s = event('e2', False) in the spec  → same logical event


# ═══════════════════════════════════════════════════════════════════════════════
# SMALL FACTORY
# ═══════════════════════════════════════════════════════════════════════════════
#
#   M1 ──e1(start)──► busy ──e2(finish)──► idle
#   M2 ──e3(start)──► busy ──e4(finish)──► idle
#   Buf: empty ──e2──► full ──e3──► empty
#        (M1 can only finish if buffer is empty; M2 can only start if buffer is full)
#
#   Controllable : e1 (start M1), e3 (start M2)
#   Conflict     : only 1 spec → no conflict possible

def build_small_factory():
    # ── States ────────────────────────────────────────────────────────────────
    m1_idle   = state('M1_idle',   marked=True)
    m1_busy   = state('M1_busy',   marked=False)
    m2_idle   = state('M2_idle',   marked=True)
    m2_busy   = state('M2_busy',   marked=False)
    buf_empty = state('Buf_empty', marked=True)
    buf_full  = state('Buf_full',  marked=False)

    # ── Plant events ──────────────────────────────────────────────────────────
    e1 = event('e1', controllable=True)    # start M1
    e2 = event('e2', controllable=False)   # M1 finishes
    e3 = event('e3', controllable=True)    # start M2
    e4 = event('e4', controllable=False)   # M2 finishes

    # ── Spec events (same names — UltraDES synchronises by name) ─────────────
    e2_s = event('e2', controllable=False)
    e3_s = event('e3', controllable=True)

    # ── Plant automata ────────────────────────────────────────────────────────
    M1  = dfa([(m1_idle, e1, m1_busy),
               (m1_busy, e2, m1_idle)], m1_idle, 'M1')
    M2  = dfa([(m2_idle, e3, m2_busy),
               (m2_busy, e4, m2_idle)], m2_idle, 'M2')

    # ── Specification automaton ───────────────────────────────────────────────
    # Buffer: M1 can only finish (e2) when buffer is empty;
    #         M2 can only start (e3) when buffer is full.
    Buf = dfa([(buf_empty, e2_s, buf_full),
               (buf_full,  e3_s, buf_empty)], buf_empty, 'Buf')

    plants      = [M1, M2]
    specs       = [Buf]
    spec_groups = [[Buf]]                         # 1 spec → 1 group
    sim_seq     = ['e1', 'e2', 'e3', 'e4']

    return plants, specs, spec_groups, sim_seq


# ═══════════════════════════════════════════════════════════════════════════════
# EXTENDED SMALL FACTORY
# ═══════════════════════════════════════════════════════════════════════════════
#
#   M1 ──a1──► busy ──b1──► idle     (a=start, b=finish)
#   M2 ──a2──► busy ──b2──► idle
#   M3 ──a3──► busy ──b3──► idle
#   B1: empty ──b1──► full ──a2──► empty  (buffer between M1 and M2)
#   B2: empty ──b2──► full ──a3──► empty  (buffer between M2 and M3)
#
#   Controllable : a1, a2, a3
#   Conflict     : B1 uses {b1,a2}, B2 uses {b2,a3} → disjoint → no conflict

def build_extended_small_factory():
    # ── States ────────────────────────────────────────────────────────────────
    m1i = state('M1_idle', True);  m1b = state('M1_busy', False)
    m2i = state('M2_idle', True);  m2b = state('M2_busy', False)
    m3i = state('M3_idle', True);  m3b = state('M3_busy', False)
    b1e = state('B1_empty', True); b1f = state('B1_full', False)
    b2e = state('B2_empty', True); b2f = state('B2_full', False)

    # ── Plant events ──────────────────────────────────────────────────────────
    a1 = event('a1', controllable=True);  b1 = event('b1', controllable=False)
    a2 = event('a2', controllable=True);  b2 = event('b2', controllable=False)
    a3 = event('a3', controllable=True);  b3 = event('b3', controllable=False)

    # ── Spec events ───────────────────────────────────────────────────────────
    b1_s = event('b1', controllable=False); a2_s = event('a2', controllable=True)
    b2_s = event('b2', controllable=False); a3_s = event('a3', controllable=True)

    # ── Plant automata ────────────────────────────────────────────────────────
    M1 = dfa([(m1i, a1, m1b), (m1b, b1, m1i)], m1i, 'M1')
    M2 = dfa([(m2i, a2, m2b), (m2b, b2, m2i)], m2i, 'M2')
    M3 = dfa([(m3i, a3, m3b), (m3b, b3, m3i)], m3i, 'M3')

    # ── Specification automata ────────────────────────────────────────────────
    B1 = dfa([(b1e, b1_s, b1f), (b1f, a2_s, b1e)], b1e, 'B1')
    B2 = dfa([(b2e, b2_s, b2f), (b2f, a3_s, b2e)], b2e, 'B2')

    plants      = [M1, M2, M3]
    specs       = [B1, B2]
    spec_groups = [[B1], [B2]]                    # disjoint alphabets → separate groups
    sim_seq     = ['a1', 'b1', 'a2', 'b2', 'a3', 'b3']

    return plants, specs, spec_groups, sim_seq


# ═══════════════════════════════════════════════════════════════════════════════
# FLEXIBLE MANUFACTURING SYSTEM  (Queiroz & Cury 2000)
# ═══════════════════════════════════════════════════════════════════════════════
#
# Plants  : C1, C2, Lathe, Mill, Robot, AM (Assembly Machine), C3, PD
# Specs   : E1 … E8  (buffer/routing constraints)
#
# Event naming convention:  XY
#   X = machine number (1=C1, 2=C2, 3=Robot, 4=Lathe, 5=Mill,
#                       6=AM, 7=C3, 8=PD)
#   Y = 1 (start, controllable), 2+ (finish, uncontrollable)
#
# Conflict resolution:
#   E7 and E8 share events through the plant (Robot, C3, PD, AM).
#   They are composed in parallel (E78 = E7 ∥ E8) before synthesis,
#   which resolves the conflict as described in Chapter 6 of the thesis.

def build_fms():
    def s(name, marked=True):
        return state(name, marked)

    def e(name, ctrl):
        return event(name, controllable=ctrl)

    # ── Plants ────────────────────────────────────────────────────────────────
    # Simple two-state machines: idle ──start──► busy ──finish──► idle

    c1_0 = s('c1_0'); c1_1 = s('c1_1', False)
    C1 = dfa([(c1_0, e('11',True), c1_1), (c1_1, e('12',False), c1_0)], c1_0, 'C1')

    c2_0 = s('c2_0'); c2_1 = s('c2_1', False)
    C2 = dfa([(c2_0, e('21',True), c2_1), (c2_1, e('22',False), c2_0)], c2_0, 'C2')

    la_0 = s('la_0'); la_1 = s('la_1', False)
    Lathe = dfa([(la_0, e('41',True), la_1), (la_1, e('42',False), la_0)], la_0, 'Lathe')

    pd_0 = s('pd_0'); pd_1 = s('pd_1', False)
    PD = dfa([(pd_0, e('81',True), pd_1), (pd_1, e('82',False), pd_0)], pd_0, 'PD')

    # Mill: two possible operations (51/52 = op A,  53/54 = op B)
    mi_0 = s('mi_0'); mi_1 = s('mi_1', False); mi_2 = s('mi_2', False)
    Mill = dfa([
        (mi_0, e('51',True),  mi_1), (mi_1, e('52',False), mi_0),
        (mi_0, e('53',True),  mi_2), (mi_2, e('54',False), mi_0),
    ], mi_0, 'Mill')

    # C3: two possible routes (71/72 = route A,  73/74 = route B)
    c3_0 = s('c3_0'); c3_1 = s('c3_1', False); c3_2 = s('c3_2', False)
    C3 = dfa([
        (c3_0, e('71',True),  c3_1), (c3_1, e('72',False), c3_0),
        (c3_0, e('73',True),  c3_2), (c3_2, e('74',False), c3_0),
    ], c3_0, 'C3')

    # Robot: five possible operations (31-39/30)
    r0=s('r0'); r1=s('r1',False); r2=s('r2',False)
    r3=s('r3',False); r4=s('r4',False); r5=s('r5',False)
    Robot = dfa([
        (r0, e('31',True), r1), (r1, e('32',False), r0),
        (r0, e('33',True), r2), (r2, e('34',False), r0),
        (r0, e('35',True), r3), (r3, e('36',False), r0),
        (r0, e('37',True), r4), (r4, e('38',False), r0),
        (r0, e('39',True), r5), (r5, e('30',False), r0),
    ], r0, 'Robot')

    # AM (Assembly Machine): 4 states, two assembly paths
    a0=s('a0'); a1=s('a1',False); a2=s('a2',False); a3=s('a3',False)
    AM = dfa([
        (a0, e('61',True),  a1),
        (a1, e('63',True),  a2), (a2, e('64',False), a0),  # path A
        (a1, e('65',True),  a3), (a3, e('66',False), a0),  # path B
    ], a0, 'AM')

    plants = [C1, C2, Lathe, Mill, Robot, AM, C3, PD]

    # ── Specifications ────────────────────────────────────────────────────────
    # Each spec is a buffer constraint: machine X deposits (uncontrollable finish)
    # then machine Y picks up (controllable start).
    # State 0 = buffer empty, State 1 = buffer full.

    # E1: C1 deposits (12) → Robot picks up (31)
    E1 = dfa([(s('E1_0'),       e('12',False), s('E1_1',False)),
              (s('E1_1',False), e('31',True),  s('E1_0'))],  s('E1_0'), 'E1')

    # E2: C2 deposits (22) → Robot picks up (33)
    E2 = dfa([(s('E2_0'),       e('22',False), s('E2_1',False)),
              (s('E2_1',False), e('33',True),  s('E2_0'))],  s('E2_0'), 'E2')

    # E3: Robot deposits (32 or 42) → Lathe or Robot picks up (41 or 35)
    e3_0=s('E3_0'); e3_1=s('E3_1',False); e3_2=s('E3_2',False)
    E3 = dfa([
        (e3_0, e('32',False), e3_1), (e3_1, e('41',True), e3_0),
        (e3_0, e('42',False), e3_2), (e3_2, e('35',True), e3_0),
    ], e3_0, 'E3')

    # E4: Robot deposits (34) → Mill picks up (51 or 53); Mill deposits → Robot picks up
    e4_0=s('E4_0'); e4_1=s('E4_1',False); e4_2=s('E4_2',False); e4_3=s('E4_3',False)
    E4 = dfa([
        (e4_0, e('34',False), e4_1),
        (e4_1, e('51',True),  e4_0), (e4_1, e('53',True),  e4_0),
        (e4_0, e('52',False), e4_2), (e4_2, e('37',True),  e4_0),
        (e4_0, e('54',False), e4_3), (e4_3, e('39',True),  e4_0),
    ], e4_0, 'E4')

    # E5: Robot deposits (36) → AM picks up (61)
    E5 = dfa([(s('E5_0'),       e('36',False), s('E5_1',False)),
              (s('E5_1',False), e('61',True),  s('E5_0'))],  s('E5_0'), 'E5')

    # E6: Robot deposits (38) → AM picks up (63)
    E6 = dfa([(s('E6_0'),       e('38',False), s('E6_1',False)),
              (s('E6_1',False), e('63',True),  s('E6_0'))],  s('E6_0'), 'E6')

    # E7: buffer B7 — two paths:
    #   path 1: Robot deposits (30) → C3 picks up (71)
    #   path 2: C3 deposits   (74) → AM  picks up (65)
    e7_0=s('E7_0'); e7_1=s('E7_1',False); e7_2=s('E7_2',False)
    E7 = dfa([
        (e7_0, e('30',False), e7_1), (e7_1, e('71',True), e7_0),
        (e7_0, e('74',False), e7_2), (e7_2, e('65',True), e7_0),
    ], e7_0, 'E7')

    # E8: buffer B8 — two paths:
    #   path 1: C3 deposits (72) → PD  picks up (81)
    #   path 2: PD deposits (82) → C3  picks up (73)
    e8_0=s('E8_0'); e8_1=s('E8_1',False); e8_2=s('E8_2',False)
    E8 = dfa([
        (e8_0, e('72',False), e8_1), (e8_1, e('81',True), e8_0),
        (e8_0, e('82',False), e8_2), (e8_2, e('73',True), e8_0),
    ], e8_0, 'E8')

    # E7 and E8 share events through the plant (Robot, C3, AM, PD).
    # Synthesising them separately would raise "Conflicting Supervisors".
    # Solution: compose them in parallel first → single combined spec E78.
    E78 = parallel_composition(E7, E8)

    specs       = [E1, E2, E3, E4, E5, E6, E78]
    spec_groups = [[E1], [E2], [E3], [E4], [E5], [E6], [E78]]

    sim_seq = ['11','12','31','32','41','42','35','36','61',
               '39','30','71','72','81','82','73','74']

    return plants, specs, spec_groups, sim_seq



# ══════════════════════════════════════════════════════════════════════════════════════
# CARGO TRANSFER DELIVERY SYSTEM  (running example)
# ══════════════════════════════════════════════════════════════════════════════════════
#
#   Three autonomous UAVs and a shared transfer buffer B:
#     D1 carries packages from the warehouse to buffer B
#     D2, D3 collect packages from B and deliver them to the client
#     B is a shared intermediate store with LIMITED CAPACITY (cap)
#
#   Event mapping (diagram alpha/beta  ->  code):
#     a1 = alpha1 : D1 starts at warehouse  (controllable)
#     b1 = beta1  : D1 deposits into B      (uncontrollable)  -> B + 1
#     a2 = alpha2 : D2 picks up from B       (controllable)    -> B - 1
#     b2 = beta2  : D2 delivers to client    (uncontrollable)
#     a3 = alpha3 : D3 picks up from B       (controllable)    -> B - 1
#     b3 = beta3  : D3 delivers to client    (uncontrollable)
#
#   The single shared buffer spec B couples b1, a2 and a3, so local-modular
#   synthesis yields ONE supervisor.  After reduction it collapses to (cap+1)
#   states -- it is exactly the buffer-level counter, which is the shared
#   resource each ESP32 node replicates in the distributed firmware.
#
#   Controllable : a1, a2, a3       Uncontrollable : b1, b2, b3
#   b1 is uncontrollable, so supC enforces the buffer-full limit by disabling
#   a1 (D1 take-off) when B is full -- the same mechanism as small_factory.

def build_delivery_system(cap=2):
    # ── UAV plants : idle --start--> busy --finish--> idle ────────────────
    d1i = state('D1_idle', True);  d1b = state('D1_busy', False)
    d2i = state('D2_idle', True);  d2b = state('D2_busy', False)
    d3i = state('D3_idle', True);  d3b = state('D3_busy', False)

    a1 = event('a1', controllable=True);  b1 = event('b1', controllable=False)
    a2 = event('a2', controllable=True);  b2 = event('b2', controllable=False)
    a3 = event('a3', controllable=True);  b3 = event('b3', controllable=False)

    D1 = dfa([(d1i, a1, d1b), (d1b, b1, d1i)], d1i, 'D1')
    D2 = dfa([(d2i, a2, d2b), (d2b, b2, d2i)], d2i, 'D2')
    D3 = dfa([(d3i, a3, d3b), (d3b, b3, d3i)], d3i, 'D3')

    # ── Shared buffer spec B (capacity `cap`) ─────────────────────
    # b1 fills (+1, blocked when full); a2/a3 consume (-1, blocked when empty).
    b1_s = event('b1', controllable=False)
    a2_s = event('a2', controllable=True)
    a3_s = event('a3', controllable=True)

    bstates = [state(f'B{k}', marked=(k == 0)) for k in range(cap + 1)]
    btrans = []
    for k in range(cap):
        btrans.append((bstates[k], b1_s, bstates[k + 1]))            # fill
    for k in range(1, cap + 1):
        btrans.append((bstates[k], a2_s, bstates[k - 1]))            # D2 consume
        btrans.append((bstates[k], a3_s, bstates[k - 1]))            # D3 consume
    B = dfa(btrans, bstates[0], 'B')

    plants      = [D1, D2, D3]
    specs       = [B]
    spec_groups = [[B]]                          # 1 shared spec -> 1 supervisor
    sim_seq     = ['a1', 'b1', 'a1', 'b1', 'a2', 'b2',
                   'a3', 'b3', 'a1', 'b1', 'a2', 'b2']
    return plants, specs, spec_groups, sim_seq


# ── Problem registry ──────────────────────────────────────────────────────────
PROBLEMS = {
    'small_factory':          build_small_factory,
    'extended_small_factory': build_extended_small_factory,
    'fms':                    build_fms,
    'delivery_system':        build_delivery_system,
}

assert PROBLEM in PROBLEMS, (
    f'Unknown problem "{PROBLEM}". '
    f'Valid options: {list(PROBLEMS.keys())}'
)
print('Problem definitions loaded.')


Problem definitions loaded.


In [ ]:
# ── Cell 5: Extraction helpers ────────────────────────────────────────────────
#
# These functions convert UltraDES DFA objects (which are .NET objects wrapped
# by pythonnet) into plain Python dicts that Cell 7 can serialise to C arrays.
#
# UltraDES-Python note:
#   All .NET properties are exposed in PascalCase:
#     automaton.Transitions  — iterable of transition objects
#     transition.Origin      — origin state object
#     transition.Trigger     — event object
#     transition.Destination — destination state object
#     automaton.InitialState — initial state object
#   Use str() to get the string name of any state or event.
#
# Determinism note:
#   UltraDES stores transitions in .NET hash-based collections, which iterate
#   in an unspecified order. We therefore sort the transition list per event
#   below so the generated .h file is reproducible across runs.


def _global_events(plants, specs):
    """
    Collect every event name that appears anywhere in plants or specs.
    Returns a sorted list — the order defines the global event index used
    throughout the generated C arrays (EVENT_NAMES, SIM_SEQ, enable, tcnt, trans).
    """
    names = set()
    for automaton in plants + specs:
        for tr in automaton.Transitions:
            names.add(str(tr.Trigger))
    return sorted(names)


def _extract_ultrades(sup, global_events, name, method):
    """
    Convert one UltraDES supervisor DFA into a plain dict for C-array generation.

    Parameters
    ----------
    sup           : UltraDES DFA object (result of monolithic_supervisor or
                    one element from local_modular_supervisors)
    global_events : sorted list of all event names (from _global_events)
    name          : label string for this supervisor (e.g. 'S0', 'Monolithic')
    method        : 'monolithic' or 'local_modular'

    Returned dict fields
    --------------------
    name            label string
    method          'monolithic' | 'local_modular'
    num_states      total number of reachable states
    num_transitions total number of transitions
    initial_state   one-hot list: initial_state[i] = 1 iff state i is the initial state
    own_events      sorted list of event names in this supervisor's alphabet
    transitions     { event_name: [[origin_idx, dest_idx], ...] }
                    only events that actually appear as transitions
                    pairs sorted by (origin_idx, dest_idx) for reproducibility
    enablement      { event_name: [0/1 per state] }
                    enable[ev][s] = 1 iff the supervisor allows ev in state s
                    (equivalently: there exists a transition on ev from state s)
    nonblocking     always True — UltraDES supC is always nonblocking
    reachable       True iff the initial state appears in the transition set
    """
    # ── Collect all states and transitions from UltraDES ──────────────────────
    state_names = set()
    raw_trans   = []   # list of (origin_name, event_name, dest_name)

    for tr in sup.Transitions:
        origin  = str(tr.Origin)
        trigger = str(tr.Trigger)
        dest    = str(tr.Destination)
        state_names.update([origin, dest])
        raw_trans.append((origin, trigger, dest))

    # ── State ordering: initial state first, rest alphabetical ────────────────
    # This ensures the one-hot initial_state vector is always [1, 0, 0, ...]
    # which simplifies reading and debugging.
    init_name = str(sup.InitialState)
    sorted_states = sorted(state_names)
    if init_name in sorted_states:
        sorted_states.remove(init_name)
        sorted_states = [init_name] + sorted_states

    state_idx = {name_: i for i, name_ in enumerate(sorted_states)}
    n = len(sorted_states)

    # ── Initial state one-hot vector ──────────────────────────────────────────
    init_vec = [0] * n
    if init_name in state_idx:
        init_vec[state_idx[init_name]] = 1

    # ── Own events: events that appear as transitions in this supervisor ───────
    own_events = sorted({ev for (_, ev, _) in raw_trans})

    # ── Transition table: event → list of [from_idx, to_idx], sorted ─────────
    # Sorting makes the generated .h file reproducible — UltraDES's iteration
    # order over its internal hash sets is otherwise unspecified.
    transitions = {}
    for (orig, ev, dest) in raw_trans:
        transitions.setdefault(ev, []).append([state_idx[orig], state_idx[dest]])
    for ev in transitions:
        transitions[ev].sort()

    # ── Enablement matrix: event → per-state 0/1 ─────────────────────────────
    # enable[ev][s] = 1 means the supervisor has a transition on ev from state s,
    # i.e. ev is enabled in state s.
    enablement = {}
    for ev in own_events:
        row = [0] * n
        for (orig, trigger, _) in raw_trans:
            if trigger == ev:
                row[state_idx[orig]] = 1
        enablement[ev] = row

    return {
        'name':            name,
        'method':          method,
        'num_states':      n,
        'num_transitions': sum(len(v) for v in transitions.values()),
        'initial_state':   init_vec,
        'own_events':      own_events,
        'transitions':     transitions,
        'enablement':      enablement,
        'nonblocking':     True,
        'reachable':       (init_name in state_names),
    }


print('Extraction helpers ready.')


In [ ]:
# ── Cell 6: Supervisor synthesis ─────────────────────────────────────────────
#
# Computes both the monolithic and local modular supervisors for the chosen problem.
#
# MONOLITHIC
# ──────────
# monolithic_supervisor(plants, specs) computes the supremal controllable
# sublanguage in a single step over the full composed plant.
# Always correct but can produce a very large supervisor for complex systems.
#
# LOCAL MODULAR
# ─────────────
# Each spec group is synthesised independently using local_modular_supervisors().
# This is valid when specs in different groups do not conflict (their combined
# supervisors are jointly nonblocking).
# The spec_groups defined in Cell 4 already encode the conflict-free grouping.
#
# The result is a list of small supervisors (lmod_list) — one per group —
# that together enforce the same behaviour as the monolithic supervisor,
# but are individually much smaller and faster to evaluate on the ESP32.
#
# Sign convention for the reduction diffs printed below:
#   negative = smaller after reduction (e.g. "(-4)" means "4 fewer states").

import traceback

try:
    # ── Build the problem ─────────────────────────────────────────────────────
    base_plants, base_specs, spec_groups, sim_seq = PROBLEMS[PROBLEM]()

    # Collect every event name across all plants and specs.
    # This defines the global event index used in the C arrays.
    gevents = _global_events(base_plants, base_specs)

    print(f'Problem       : {PROBLEM}')
    print(f'Plants        : {[g.Name for g in base_plants]}')
    print(f'Specs         : {[s.Name for s in base_specs]}')
    print(f'Spec groups   : {[[s.Name for s in g] for g in spec_groups]}')
    print(f'Global events : {gevents}')
    print()

    # ── Monolithic supervisor ─────────────────────────────────────────────────
    print('Computing monolithic supervisor...')
    mono_sup = monolithic_supervisor(base_plants, base_specs)

    # Compose supervisor with the full plant to count closed-loop transitions.
    plant_composition = base_plants[0]
    for p in base_plants[1:]:
        plant_composition = parallel_composition(plant_composition, p)
    mono_closed = parallel_composition(mono_sup, plant_composition)

    mono_dict = _extract_ultrades(mono_sup, gevents, 'Monolithic', 'monolithic')
    print(f'  States              : {mono_dict["num_states"]}')
    print(f'  Transitions (sup)   : {len(list(mono_sup.Transitions))}')
    print(f'  Transitions (S||G)  : {len(list(mono_closed.Transitions))}')
    print(f'  Initial reachable   : {mono_dict["reachable"]}')
    print()

    # ── Local modular supervisors ─────────────────────────────────────────────
    print('Computing local modular supervisors...')
    lmod_list = []
    sup_idx   = 0

    for g_idx, group in enumerate(spec_groups):
        group_names = [s.Name for s in group]
        print(f'  Group {g_idx}: {group_names}')

        # Synthesise one supervisor per spec in this conflict-free group.
        group_sups = local_modular_supervisors(base_plants, group)

        for sup in group_sups:
            label = f'S{sup_idx}'
            d = _extract_ultrades(sup, gevents, label, 'local_modular')
            lmod_list.append(d)
            print(f'    {label}: {d["num_states"]:>4} states  '
                  f'{d["num_transitions"]:>4} transitions  '
                  f'own events: {d["own_events"]}')
            sup_idx += 1

    print()
    print('Synthesis complete.')
    print(f'  Monolithic  : 1 supervisor, {mono_dict["num_states"]} states, '
          f'{mono_dict["num_transitions"]} transitions')
    print(f'  Local mod   : {len(lmod_list)} supervisors, '
          f'{sum(d["num_states"] for d in lmod_list)} states total, '
          f'{sum(d["num_transitions"] for d in lmod_list)} transitions total')

    # ── Local modular REDUCED supervisors ────────────────────────────────────
    # local_modular_reduced_supervisors() applies supervisor reduction
    # (Malik et al. 2004) after synthesis — finds the smallest control-
    # congruent DFA for each supervisor.  Called sequentially (one group at
    # a time) to avoid race conditions inside UltraDES's internal Parallel.ForEach.
    print('Computing local modular reduced supervisors...')
    lmod_red_list = []
    sup_idx = 0

    for g_idx, group in enumerate(spec_groups):
        group_names = [s.Name for s in group]
        print(f'  Group {g_idx}: {group_names}')

        group_red_sups = local_modular_reduced_supervisors(base_plants, group)

        for sup in group_red_sups:
            label = f'S{sup_idx}_red'
            d     = _extract_ultrades(sup, gevents, label, 'local_modular_reduced')
            lmod_red_list.append(d)
            orig  = lmod_list[sup_idx] if sup_idx < len(lmod_list) else None
            # Negative delta = smaller after reduction.
            ds    = (d['num_states']      - orig['num_states'])      if orig else 0
            dt    = (d['num_transitions'] - orig['num_transitions']) if orig else 0
            print(f'    {label}: {d["num_states"]:>4} states ({ds:+d})  '
                  f'{d["num_transitions"]:>4} transitions ({dt:+d})  '
                  f'own events: {d["own_events"]}')
            sup_idx += 1

    print()
    print(f'  Local mod red : {len(lmod_red_list)} supervisors, '
          f'{sum(d["num_states"] for d in lmod_red_list)} states total, '
          f'{sum(d["num_transitions"] for d in lmod_red_list)} transitions total')

except Exception as exc:
    print(f'Synthesis error: {exc}')
    traceback.print_exc()


In [ ]:
# ── Cell 7: Generate supervisor_data_<PROBLEM>.h ─────────────────────────────
#
# OUTPUT FORMAT — direct C arrays, no JSON, no runtime parsing
# ─────────────────────────────────────────────────────────────
# The .h file embeds all supervisor data as typed C arrays in PROGMEM (flash).
# The ESP32 firmware reads them directly with pgm_read_byte / pgm_read_word —
# zero heap cost, zero parse time.
#
# ARRAY LAYOUT
# ────────────
# For each supervisor (prefix = 'lm0_', 'lm1_', ..., 'mono_'):
#
#   <prefix>init[num_states]          int8_t   one-hot initial state vector
#   <prefix>enable[EVENT_COUNT * n]   int8_t   flat enablement matrix:
#                                               enable[event_gi * n + state] = 0/1
#   <prefix>tcnt[EVENT_COUNT]         uint16_t number of (from,to) pairs per event
#   <prefix>trans[2 * sum(tcnt)]      int16_t  flat transition pairs: (from,to,from,to,...)
#                                               grouped by event in global event order
#
# SupDesc struct (defined in the .h):
#   Links name + num_states + pointers to the four arrays above.
#   The firmware iterates LMOD_SUPS[0..LMOD_COUNT-1] at runtime.
#
# FLASH SIZE CHECK
# ────────────────
# The monolithic supervisor is omitted when its actual C-array footprint
# exceeds MONO_FLASH_LIMIT. The firmware detects this via the HAS_MONO macro.

import os

# ── Output filename ───────────────────────────────────────────────────────────
OUTPUT_H = f'supervisor_data_{PROBLEM}.h'

# Monolithic supervisor flash limit (bytes).
# Default 1.2 MB matches the default ESP32 APP partition size.
# Increase if you use a custom partition with a larger sketch region.
MONO_FLASH_LIMIT = 1_200_000


# ═════════════════════════════════════════════════════════════════════════════
# C ARRAY HELPERS
# ═════════════════════════════════════════════════════════════════════════════

def c_array_i8(name, values, per_row=20):
    """Emit a PROGMEM int8_t array (values are 0 or 1)."""
    rows = [values[i:i+per_row] for i in range(0, len(values), per_row)]
    body = ',\n    '.join(', '.join(str(v) for v in r) for r in rows)
    return f'static const int8_t {name}[] PROGMEM = {{\n    {body}\n}};\n'

def c_array_i16(name, values, per_row=16):
    """Emit a PROGMEM int16_t array (transition from/to state indices)."""
    rows = [values[i:i+per_row] for i in range(0, len(values), per_row)]
    body = ',\n    '.join(', '.join(str(v) for v in r) for r in rows)
    return f'static const int16_t {name}[] PROGMEM = {{\n    {body}\n}};\n'

def c_array_u16(name, values, per_row=20):
    """Emit a PROGMEM uint16_t array (transition counts or sequence indices)."""
    rows = [values[i:i+per_row] for i in range(0, len(values), per_row)]
    body = ',\n    '.join(', '.join(str(v) for v in r) for r in rows)
    return f'static const uint16_t {name}[] PROGMEM = {{\n    {body}\n}};\n'


# ═════════════════════════════════════════════════════════════════════════════
# FLASH-SIZE ESTIMATOR
# ═════════════════════════════════════════════════════════════════════════════
#
# Returns the EXACT byte count that this supervisor's four C arrays will
# occupy in PROGMEM — matches what the linker actually places in flash.

def supervisor_flash_bytes(d, num_events):
    n  = d['num_states']
    tc = sum(len(pairs) for pairs in d['transitions'].values())   # total (from,to) pairs
    return (
        n               +   # init      [n]            × int8_t
        num_events * n  +   # enable    [ne * n]       × int8_t
        num_events * 2  +   # tcnt      [ne]           × uint16_t
        tc * 4              # trans     [2 * tc]       × int16_t
    )


# ═════════════════════════════════════════════════════════════════════════════
# SUPERVISOR EMITTER
# ═════════════════════════════════════════════════════════════════════════════

def emit_supervisor(d, prefix, gevents):
    """
    Emit all four C arrays for one supervisor.

    Parameters
    ----------
    d        : supervisor dict from _extract_ultrades()
    prefix   : C identifier prefix, e.g. 'lm0_' or 'mono_'
    gevents  : global event list (defines array layout order)

    Returns
    -------
    (c_code : str, flash_bytes : int)
    """
    n  = d['num_states']
    ne = len(gevents)
    out = []

    # ── 1. Initial state one-hot vector  [n] × int8_t ─────────────────────────
    out.append(c_array_i8(f'{prefix}init', d['initial_state']))

    # ── 2. Enablement matrix  [ne × n] × int8_t ───────────────────────────────
    # Flat layout: enable[event_gi * n + state_idx]
    # If an event does not appear in this supervisor's alphabet, its entire row
    # is all-ones (that event is always enabled — no constraint from this sup).
    enable_flat = []
    for gev in gevents:
        row = d['enablement'].get(gev, None)
        enable_flat.extend(row if row is not None else [1] * n)
    out.append(c_array_i8(f'{prefix}enable', enable_flat))

    # ── 3. Transition pair counts  [ne] × uint16_t ────────────────────────────
    # tcnt[gi] = number of (from,to) pairs for global event gi.
    # Used by the firmware to find the start offset of each event's block in trans[].
    trans_flat   = []
    trans_counts = []
    for gev in gevents:
        pairs = d['transitions'].get(gev, [])
        trans_counts.append(len(pairs))
        for (orig, dest) in pairs:
            trans_flat.extend([orig, dest])   # interleaved: from0,to0,from1,to1,...
    out.append(c_array_u16(f'{prefix}tcnt', trans_counts))

    # ── 4. Transition pairs  [2 × sum(tcnt)] × int16_t ────────────────────────
    if trans_flat:
        out.append(c_array_i16(f'{prefix}trans', trans_flat))
    else:
        # No transitions at all — emit a placeholder so the array is never empty.
        out.append(f'static const int16_t {prefix}trans[] PROGMEM = {{0}};'
                   f'  // no transitions\n')

    return '\n'.join(out), supervisor_flash_bytes(d, ne)


# ═════════════════════════════════════════════════════════════════════════════
# FLASH SIZE CHECK
# ═════════════════════════════════════════════════════════════════════════════

print('Flash size estimates (exact C-array bytes):')
mono_flash_estimate = supervisor_flash_bytes(mono_dict, len(gevents))
print(f'  Monolithic  : {mono_flash_estimate:>10,} bytes')
print(f'  Limit       : {MONO_FLASH_LIMIT:>10,} bytes')

include_mono = (mono_flash_estimate <= MONO_FLASH_LIMIT)
print(f'  Decision    : {"INCLUDED" if include_mono else "OMITTED — exceeds MONO_FLASH_LIMIT"}')
print()


# ═════════════════════════════════════════════════════════════════════════════
# BUILD THE .h FILE
# ═════════════════════════════════════════════════════════════════════════════

lines = []

# ── File header ───────────────────────────────────────────────────────────────
lines += [
    f'// {OUTPUT_H}',
    '// Auto-generated by the UltraDES notebook — DO NOT EDIT BY HAND.',
    '// Re-run Cell 7 to regenerate.',
    '//',
    f'// Problem   : {PROBLEM}',
    f'// Events    : {gevents}',
    f'// Sim seq   : {sim_seq}',
    f'// Monolithic: {"INCLUDED" if include_mono else "OMITTED (exceeds MONO_FLASH_LIMIT)"}',
    '',
    '#pragma once',
    '#include <stdint.h>',
    '// pgmspace.h is provided automatically by the ESP32 Arduino core.',
    '',
]

# ── Global constants ──────────────────────────────────────────────────────────
ne = len(gevents)
lines += [
    f'// Total number of events across all plants and specs.',
    f'#define EVENT_COUNT {ne}',
    '',
    f'// Number of steps in the simulation sequence.',
    f'#define SIM_SEQ_LEN {len(sim_seq)}',
    '',
]

# ── Event name strings (PROGMEM) + pointer array ─────────────────────────────
lines.append('// Event names in global index order (index = position in EVENT_NAMES[]).')
for gi, ev in enumerate(gevents):
    lines.append(f'static const char EV_{gi}[] PROGMEM = "{ev}";')
lines += [
    'static const char* const EVENT_NAMES[] PROGMEM = {',
    '    ' + ', '.join(f'EV_{gi}' for gi in range(ne)),
    '};',
    '',
]

# ── Simulation sequence as event indices ─────────────────────────────────────
sim_idx = [gevents.index(ev) for ev in sim_seq]
lines.append('// Simulation sequence: each entry is a global event index into EVENT_NAMES[].')
lines.append(c_array_u16('SIM_SEQ', sim_idx))
lines.append('')

# ── SupDesc struct ────────────────────────────────────────────────────────────
lines += [
    '// Descriptor struct — one instance per supervisor.',
    '// All pointer fields point into PROGMEM arrays defined below.',
    'struct SupDesc {',
    '    const char*      name;       // supervisor label (PROGMEM string)',
    '    uint16_t         num_states; // total number of states',
    '    const int8_t*    init;       // one-hot initial state vector [num_states]',
    '    const int8_t*    enable;     // enablement matrix [EVENT_COUNT * num_states]',
    '    const uint16_t*  tcnt;       // transition pair counts per event [EVENT_COUNT]',
    '    const int16_t*   trans;      // flat (from,to) transition pairs',
    '};',
    '',
]

# ── Local modular supervisors ─────────────────────────────────────────────────
lines.append(f'// ── Local modular supervisors ({len(lmod_list)} total) ──────────────────────────────────')
lines.append(f'#define LMOD_COUNT {len(lmod_list)}')
lines.append('')

total_lmod_flash = 0
for idx, d in enumerate(lmod_list):
    prefix = f'lm{idx}_'
    lines.append(f'// {d["name"]}  —  {d["num_states"]} states, {d["num_transitions"]} transitions')
    code, fb = emit_supervisor(d, prefix, gevents)
    lines.append(code)
    total_lmod_flash += fb
    lines.append(f'static const char LMOD_{idx}_NAME[] PROGMEM = "{d["name"]}";')
    lines.append('')

# SupDesc array for local modular
lines.append('static const SupDesc LMOD_SUPS[] = {')
for idx, d in enumerate(lmod_list):
    prefix = f'lm{idx}_'
    lines.append(f'    // {d["name"]}: {d["num_states"]} states')
    lines.append(f'    {{ LMOD_{idx}_NAME, {d["num_states"]}, '
                 f'{prefix}init, {prefix}enable, {prefix}tcnt, {prefix}trans }},')
lines.append('};')
lines.append('')


# ── Local modular REDUCED supervisors ────────────────────────────────────────
lines.append(f'// ── Local modular reduced supervisors ({len(lmod_red_list)} total) ──────────────────────')
lines.append(f'#define LMOD_RED_COUNT {len(lmod_red_list)}')
lines.append('')

total_lmod_red_flash = 0
for idx, d in enumerate(lmod_red_list):
    prefix = f'lmr{idx}_'
    orig   = lmod_list[idx] if idx < len(lmod_list) else None
    # Negative delta = smaller after reduction.
    ds     = (d['num_states']      - orig['num_states'])      if orig else 0
    dt     = (d['num_transitions'] - orig['num_transitions']) if orig else 0
    lines.append(f'// {d["name"]}  —  {d["num_states"]} states ({ds:+d}), {d["num_transitions"]} transitions ({dt:+d})')
    code, fb = emit_supervisor(d, prefix, gevents)
    lines.append(code)
    total_lmod_red_flash += fb
    lines.append(f'static const char LMOD_RED_{idx}_NAME[] PROGMEM = "{d["name"]}";')
    lines.append('')

lines.append('static const SupDesc LMOD_RED_SUPS[] = {')
for idx, d in enumerate(lmod_red_list):
    prefix = f'lmr{idx}_'
    lines.append(f'    // {d["name"]}: {d["num_states"]} states')
    lines.append(f'    {{ LMOD_RED_{idx}_NAME, {d["num_states"]}, '
                 f'{prefix}init, {prefix}enable, {prefix}tcnt, {prefix}trans }},')
lines.append('};')
lines.append('')

# ── Monolithic supervisor (conditional) ──────────────────────────────────────
if include_mono:
    lines.append(f'// ── Monolithic supervisor — {mono_dict["num_states"]} states ─────────────────────────')
    code, _ = emit_supervisor(mono_dict, 'mono_', gevents)
    lines.append(code)
    lines += [
        f'static const char MONO_NAME[] PROGMEM = "{mono_dict["name"]}";',
        f'#define MONO_NUM_STATES {mono_dict["num_states"]}',
        'static const SupDesc MONO_SUP = {',
        f'    MONO_NAME, {mono_dict["num_states"]}, '
        f'mono_init, mono_enable, mono_tcnt, mono_trans',
        '};',
        '#define HAS_MONO 1',
    ]
else:
    lines += [
        '// Monolithic supervisor omitted — exceeds MONO_FLASH_LIMIT.',
        '// The firmware will skip the monolithic benchmark automatically.',
        '#define HAS_MONO 0',
    ]
lines.append('')

# ── Write file ────────────────────────────────────────────────────────────────
h_content = '\n'.join(lines) + '\n'
with open(OUTPUT_H, 'w') as f:
    f.write(h_content)
print(f'Written : {OUTPUT_H}  ({os.path.getsize(OUTPUT_H):,} bytes on disk)')
print(f'  Local modular flash         : {total_lmod_flash:,} bytes')
print(f'  Local modular reduced flash : {total_lmod_red_flash:,} bytes')
if include_mono:
    print(f'  Monolithic flash            : {mono_flash_estimate:,} bytes')
print()

print('Next steps:')
print(f'  1. Copy {OUTPUT_H} to your Arduino sketch folder.')
print(f'  2. In the .ino, change the #include to: #include "{OUTPUT_H}"')
print( '  3. Flash the sketch.')
